In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
# CRITICAL (Error 020): prevents stale module cache.
# %matplotlib inline sets the Jupyter-native rendering backend so all
# plt.show() / IPython.display calls render figures directly in the notebook.


# 🍄 SPORE+
## Systematic Preprocessing and Optimization for Robust Evaluation Plus

In [ ]:
# ── Primary Imports & Setup ────────────────────────────────────────────────
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Add src/ to path
sys.path.insert(0, os.path.abspath("."))
import src

from src.utils import (load_config, setup_logger, apply_sporeplus_style,
                       snapshot, log_memory)
from src import plotting as splt

# CRITICAL: use sys.executable for any pip installs (Error 014)
# import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "harmonypy"])
print("✓ Imports complete")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
cfg    = load_config("sporeplus_config.yaml")
logger = setup_logger(cfg)

logger.info("SPORE+ pipeline initialized")
logger.info(f"  Dataset : {cfg['dataset']['name']}")
logger.info(f"  Input   : {cfg['paths']['_raw_h5ad']}")
logger.info(f"  Output  : {cfg['paths']['_processed']}")

# Pipeline tracker: cell counts through each phase
pipeline_tracker = {}
warnings_log     = []   # collect all warning strings for the final report

---
### Phase 0: Data Ingestion & Sparsification
This phase handles the foundational ingestion and memory optimization of the raw single-cell dataset. Phase 0 dynamically reads the raw `.h5ad` file from disk in out-of-core chunks, mathematically compresses the data into a Compressed Sparse Row (CSR) matrix by dropping the zeros, and caches the optimized file. This reduces the RAM footprint and  accelerates all downstream I/O operations.

In [ ]:
# ─── CELL 1: COMPUTATION ───────────────────────────────────────────────────
from src.phase00_sparse_convert import run_phase0

adata = run_phase0(cfg, logger)
pipeline_tracker["Phase 0: Ingested"] = adata.n_obs
snapshot(adata, "Phase 0 complete", logger)

In [ ]:
# ─── CELL 2: PLOTTING ──────────────────────────────────────────────────────
import importlib
import src.plotting as splt

# Reload the plotting module in case adjustments are made
importlib.reload(splt)

fig, path = splt.plot_sparsity_overview(adata, cfg)

In [ ]:
# ─── CELL 3: DIAGNOSTIC REPORT ─────────────────────────────────────────────
import src.diagnostics as diag
import importlib

importlib.reload(diag)

# Print the Phase 0 Summary
diag.report_phase0_diagnostics(adata)

---
## Phase 1 · Detection
Phase 1 performs non-destructive, read-only interrogation of the `.h5ad` file. It identifies the data modality (routing ATAC peaks or ADT antibodies to safe storage if Multiome/CITE-seq is detected), scans for combinatorial perturbation structures (e.g., dual-guide multiplexing), audits the `.obs` metadata for existing cell line labels, and detects the gene nomenclature format. If non-standard gene IDs (Ensembl or Entrez) are found, it queries the `mygene` database to harmonize them into standard HGNC symbols, ensuring downstream target filtering does not silently fail.


In [ ]:
# ─── CELL 1: COMPUTATION ───────────────────────────────────────────────────
import src.phase01_detection as p1
import importlib

importlib.reload(p1)

# Run Phase 1: Metadata Interrogation & Harmonization
raw_adata, detection_results = p1.run_phase1(adata, cfg, logger)

In [ ]:
# ─── CELL 2: DIAGNOSTIC REPORT ─────────────────────────────────────────────
import src.diagnostics as diag
import importlib

importlib.reload(diag)

# Print the Phase 1 Summary
diag.report_phase1_diagnostics(raw_adata, detection_results)

---
## Phase 2 · Cell-Level Triage

In [ ]:
import src.phase02_cell_triage as p2
import importlib

importlib.reload(p2)

# Run Phase 2: Cell-Level Triage
adata, p2_waterfall = p2.run_phase2(adata, cfg, logger)

In [ ]:
from src import plotting as splt
import importlib
importlib.reload(splt)

fig, path = splt.plot_qc_violin(adata, cfg)
plt.show()

In [ ]:
from src import plotting as splt
import importlib
importlib.reload(splt)

fig, path = splt.plot_mt_scatter(adata, cfg)
plt.show()

In [ ]:
from src import diagnostics as diag
import importlib

importlib.reload(diag)

# Print the Phase 2 Cell Triage Summary
diag.report_phase2_diagnostics(p2_waterfall)

---
## Phase 3 · Ambient RNA Decontamination

In [ ]:
from src.phase03_ambient import run_phase3 

adata = run_phase3(adata, cfg, logger)

pipeline_tracker["Phase 3: After ambient filter"] = adata.n_obs

In [ ]:
from src import plotting as splt

import importlib
importlib.reload(splt)

fig, path = splt.plot_ambient_diagnostics(adata, cfg)

In [ ]:
import src.diagnostics as diag
import importlib

importlib.reload(diag)

# Print the Phase 3 Ambient Summary
diag.report_phase3_diagnostics(adata, cfg)

---
## Phase 4 · Doublet Detection

In [ ]:
from src.phase04_doublets import run_phase4

adata = run_phase4(adata, cfg, logger)
pipeline_tracker["Phase 4: After doublet filter"] = adata.n_obs

In [ ]:
# Phase 4: Doublet Detection Plots
from src import plotting as splt
import importlib
importlib.reload(splt)

fig, path = splt.plot_doublets(adata, cfg)
plt.show()

In [ ]:
import src.diagnostics as diag
import importlib

importlib.reload(diag)

# Print the Phase 4 Doublet Summary
diag.report_phase4_diagnostics(adata, cfg)

---
## Phase 5 · Escaper Filtering

In [ ]:
import src.phase05_escaper_filtering as p5
import importlib

importlib.reload(p5)

# Run Phase 5: Escaper Filtering
adata, p5_escaper_stats, p5_pert_sizes = p5.run_phase5(adata, cfg, logger)

In [ ]:
# Phase 5: Perturbation Efficiency Dashboard
from src import plotting as splt
import importlib
importlib.reload(splt)

fig, path = splt.plot_perturbation_efficiency(adata, cfg)
plt.show()

In [ ]:
import src.diagnostics as diag
import importlib

importlib.reload(diag)

# Print the Phase 5 Escaper Summary
diag.report_phase5_diagnostics(adata, p5_escaper_stats, p5_pert_sizes)

---
## Phase 6 · Gene-Level Triage

In [ ]:
import src.phase06_gene_triage as p6
import importlib

importlib.reload(p6)

# Run Phase 6: Gene-Level Triage (this creates the variables)
adata, p6_waterfall, p6_rescued = p6.run_phase6(adata, cfg, logger)

In [ ]:
import src.plotting as splt
import importlib

importlib.reload(splt)

# Generate Phase 6 Diagnostic Plot & Sparsity Report
fig, path = splt.plot_phase6_diagnostics(adata, cfg)

In [ ]:
import src.diagnostics as diag
import importlib

importlib.reload(diag)

# Print the comprehensive Phase 6 Gene Triage Summary
diag.report_phase6_diagnostics(adata, p6_waterfall, p6_rescued)

---
## Phase 7 · Data Splits
*(Data leakage wall: all downstream phases fit on train only)*

In [ ]:
import src.phase07_splits as p7
import importlib

importlib.reload(p7)

# Run Phase 7: Data Splits (Destructive Memory Protocol)
p7_splits = p7.run_phase7(adata, cfg, logger)

In [ ]:
from src import plotting as splt
import importlib

importlib.reload(splt)

# THE FIX: Changed 'split_result' to 'p7_splits'
fig, path = splt.plot_phase7_splits(p7_splits, cfg)
plt.show()

In [ ]:
import src.diagnostics as diag
import importlib

importlib.reload(diag)

# Print the Phase 7 Split Summary & Leakage Checks
diag.report_phase7_diagnostics(p7_splits)

---
## Phase 8 · Dimensionality Reduction (HVG)
*Computed on train only — applied to val/test to prevent leakage.*

In [ ]:
import sys
import importlib
import src.phase08_hvg as p8

# 1. Nuke the cached Python module and force a disk reload
importlib.reload(p8)
print("✓ Phase 8 module reloaded successfully.")

# 2. DEFINITION FIX: Retrieve the splits dictionary from your tracker
splits = p7_splits[0]

# 3. Re-run Phase 8
print("Calculating stratified HVGs on Training Set...")
splits, hvg_names = p8.run_phase8(splits, cfg, logger)

# 4. Update your tracking dictionary
p7_splits[0] = splits
print("✓ Phase 8 complete. Ready for Phase 9.")

In [ ]:
print('test')

In [ ]:
import importlib
import src.plotting as splt

# Reload the plotting module to get the fix
importlib.reload(splt)

# Render the Mako Dispersion Plot
fig, path = splt.plot_phase8_hvg(splits["train"], cfg)
plt.show()

In [ ]:
import src.diagnostics as diag
import importlib

importlib.reload(diag)

# Print the Phase 8 HVG Summary
# We pass splits["train"] because that is where the hvg_stats are stored
diag.report_phase8_diagnostics(splits["train"])

---
## Phase 9 · Normalization

In [ ]:
import importlib
import src.phase09_normalization as p9
from src.phase10_confounders import force_heap_return

importlib.reload(p9)

# 1. Extract the active dictionary from Phase 7 (NO all_splits!)
splits_dict = p7_splits[0]

# 2. Run the In-Place Normalization
splits = p9.run_phase9(splits_dict, cfg, logger)

# 3. Extract lightweight plotting data before the memory wipe
print("Extracting plot cache...")
p9_cache = p9.extract_p9_data(splits)

# ─── MEMORY FIREWALL ─────────────────────────────────────────────────────────
dataset_name = cfg["dataset"]["name"]
splits_dir   = cfg["paths"]["splits_dir"] 

print("Saving normalized splits to disk...")
for key in ["train", "val", "test"]:
    if key in splits:
        path = f"{splits_dir}/{dataset_name}_{key}_p9.h5ad"
        splits[key].write_h5ad(path)
        logger.info(f"  {key} (post-P9) → {path}")

# Aggressive cleanup: nullify .X + IPython cache + malloc_trim
heap_floor = force_heap_return(
    splits_dict=splits,
    var_names=["splits", "splits_dict", "p7_splits", "adata", "train", "val", "test"],
    logger=logger)

# Nuke the dictionary from RAM to make room for Phase 10
splits = {}
logger.info(f"  Heap floor before Phase 10: {heap_floor:.1f} GB")

In [ ]:
from src import plotting as splt
import importlib

importlib.reload(splt)

# Generate Phase 9 Normalization Dashboard
fig, path = splt.plot_phase9_normalization(p9_cache, cfg)

In [ ]:
import src.diagnostics as diag
import importlib

importlib.reload(diag)

# Print the Phase 9 Normalization Summary
diag.report_phase9_diagnostics(p9_cache, cfg)

---
## Phase 10 · Confounder Mitigation
*(Cell cycle regression → Scaling → PCA → Harmony batch correction)*

In [ ]:
import importlib
import src.phase10_confounders as p10
import src.plotting as splt

importlib.reload(p10)
importlib.reload(splt)
    
# 1. Run the heavy compute in the isolated subprocess
# We pass 'None' because the matrix was wiped from RAM and is read from disk
p10_results = p10.run_phase10(None, cfg, logger)

logger.info("Phase 10 complete. Parent has embeddings only.")

In [ ]:
from src import plotting as splt
import importlib

importlib.reload(splt)

# Generate Phase 10 Integration Dashboard
# We pass the train embedding dictionary generated by the Phase 10 computation cell
fig, path = splt.plot_phase10_integration(p10_results["train"], cfg)

In [ ]:
import src.diagnostics as diag
import importlib

importlib.reload(diag)

# Print the Phase 10 Integration Summary
diag.report_phase10_diagnostics(p10_results["train"], cfg)

---
## Phase 11 · Cell Line Detection & Separation
*Runs on the full post-Harmony embedding before metacell aggregation.*

In [ ]:
import importlib
import src.phase11_cell_line_separation as p11

importlib.reload(p11)

# 1. Run Phase 11 purely on the Train embeddings
p10_results["train"], cell_line_meta = p11.run_phase11(p10_results["train"], cfg, logger)

n_cell_lines = cell_line_meta.get("n_cell_lines", 1)

# 2. Log Warnings
if n_cell_lines > 1:
    logger.warning(
        f"Phase 11: {n_cell_lines} cell line(s) detected. "
        f"Phase 12 will produce separate metacell outputs per cell line.")

In [ ]:
import src.plotting as splt
import importlib

importlib.reload(splt)

fig_2d, path_2d = splt.plot_cell_line_pca(p10_results["train"], cfg)

In [ ]:
import src.plotting as splt
import importlib

importlib.reload(splt)

# 2. Render the Interactive 3D Plotly Map
fig_3d, path_3d = splt.plot_cell_line_pca_3d(p10_results["train"], cfg)

In [ ]:
import src.diagnostics as diag
import importlib

importlib.reload(diag)

# Print the Phase 11 Cell Line Summary & Driver Gene Extractor
diag.report_phase11_diagnostics(p10_results["train"], cell_line_meta, cfg)

---
## Phase 12 · Metacell Aggregation
*Produces `{dataset}_{cell_line}_{split}_metacell.h5ad` per cell line × split.*

In [ ]:
import importlib
import gc
import numpy as np
import anndata as ad
import src.phase12_metacell as p12

importlib.reload(p12)

# Patch anndata to sanitise uns before write
p12.patch_anndata_write()

dataset_name = cfg["dataset"]["name"]
splits_dir   = cfg["paths"]["_splits"]
output_dir   = cfg["paths"]["_processed"]

logger.info("Loading cleaned Phase 10 matrices (Ghost-Free) from disk...")

# ── THE FIX: Load the _p10 matrices to maintain the strict 5000-gene envelope ──
train_p10 = ad.read_h5ad(splits_dir / f"{dataset_name}_train_p10.h5ad")
val_p10   = ad.read_h5ad(splits_dir / f"{dataset_name}_val_p10.h5ad")
test_p10  = ad.read_h5ad(splits_dir / f"{dataset_name}_test_p10.h5ad")

# ── Attach Harmony Embeddings to Train ──
embed_path = output_dir / f"{dataset_name}_train_p10_embed.npz"
emb = np.load(embed_path)
train_p10.obsm["X_pca_harmony"] = emb["X_pca_harmony"].astype("float32")
train_p10.obsm["X_pca"]         = emb["X_pca"].astype("float32")
del emb

splits_p10 = {"train": train_p10, "val": val_p10, "test": test_p10}

# ── Propagate cell line labels & Run Aggregation ──
p12.apply_cell_line_labels_to_splits(splits_p10, cell_line_meta, cfg, logger)
all_meta_splits = p12.run_phase12(splits_p10, cfg, logger)

# ── Memory Cleanup ──
del train_p10, val_p10, test_p10, splits_p10
gc.collect()

In [ ]:
import importlib
import src.plotting as splt

# Reload the plotting module
importlib.reload(splt)

# Generate the dashboard (no hardcoding required)
fig, path = splt.plot_phase12_metacell_dashboard(all_meta_splits, cfg)
if fig:
    plt.show()

In [ ]:
import importlib
import src.plotting as splt

# Reload the plotting module
importlib.reload(splt)

# Generate the QC Size Distribution Plot
fig, path = splt.plot_phase12_size_distribution(all_meta_splits, cfg)
if fig:
    plt.show()

In [ ]:
import src.diagnostics as diag
import importlib

importlib.reload(diag)

# Print the Phase 12 Metacell Condensation Summary
diag.report_phase12_diagnostics(all_meta_splits)

---
## Phase 13 · CHITIN Correction
*Runs independently per cell line. Each gets its own calibrated KNN manifold.*

In [ ]:
import src.phase13_chitin as p13
import importlib

importlib.reload(p13)

# 1. Run CHITIN Correction across all cell lines
all_chitin = p13.run_phase13(all_meta_splits, cfg, logger)

logger.info("Phase 13 computation complete.")

In [ ]:
import importlib
import src.plotting as splt

# Reload to ensure Jupyter sees the new grid function
importlib.reload(splt)

# Pass the entire all_chitin dictionary
fig_pareto, path_pareto = splt.plot_chitin_pareto_grid(all_chitin, cfg)
if fig:
    plt.show()

In [ ]:
import src.diagnostics as diag
import importlib

importlib.reload(diag)

# Print the Phase 13 Pareto Calibration Summary
diag.report_phase13_diagnostics(all_chitin)

In [ ]:
import importlib
import src.plotting as splt

importlib.reload(splt)

# Print the text summary
fig_biv, path_biv = splt.plot_chitin_bivariate_grid(all_chitin, cfg)

In [ ]:
import src.diagnostics as diag
import importlib

importlib.reload(diag)

# Print the Post-CHITIN PCA Driver Gene Shift Analysis
diag.report_phase13_pca_drivers(all_chitin)

---
## Pipeline Summary Waterfall

In [ ]:
import src.diagnostics as diag
import src.plotting as splt
import importlib

importlib.reload(diag)
importlib.reload(splt)

# 1. Dynamically calculate attrition from the files on disk
dynamic_tracker = diag.build_pipeline_tracker_from_disk(cfg, logger)

# 2. Render the Waterfall Plot
if dynamic_tracker:
    fig_waterfall, path_waterfall = splt.plot_pipeline_waterfall(dynamic_tracker, cfg)
else:
    logger.error("Tracker is empty. Could not find SPORE+ milestone files on disk.")

---
## Final Report

In [ ]:
from src.reporting import generate_report

report_path = generate_report(
    cfg              = cfg,
    pipeline_tracker = pipeline_tracker,
    detection_results= detection_results,
    cell_line_meta   = cell_line_meta,
    chitin_summary   = chitin_summary,
    all_meta_splits  = all_meta_splits,
    warnings_log     = warnings_log,
)
print(f"\n✅ SPORE+ pipeline complete. Report saved to {report_path}")
print(f"\nOutput files ready for FUNGI → SPECTRA downstream processing.")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  POST-RUN CLEANUP  (optional)
#  Set output.cleanup_intermediates: true in sporeplus_config.yaml to enable.
#  Removes: *_p9.h5ad, *_p10.h5ad, *_p10_embed.npz, checkpoints/ shards.
#  Keeps: *_metacell.h5ad, chitin outputs, split indices, figures, report.
# ════════════════════════════════════════════════════════════════════════════
from src.cleanup import run_cleanup

cleanup_result = run_cleanup(cfg, logger)
if cleanup_result["deleted"]:
    logger.info(f"  Freed {cleanup_result['total_mb_freed']:.0f} MB of intermediate files")


In [ ]:
import numpy as np
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import src.phase09_normalization as p9
import src.plotting as splt
import importlib

# Reload modules to ensure you have the latest code
importlib.reload(p9)
importlib.reload(splt)

print("⚙️ Generating 1,000-cell mock dataset to safely test Phase 9 code...")
# Create a fake sparse-ish dataset that mimics scRNA-seq counts
X_mock = np.random.negative_binomial(1, 0.1, size=(1000, 500)).astype(np.float32)
adata_mock = ad.AnnData(X=X_mock)

# Mock a splits dictionary and a config
mock_splits = {"train": adata_mock.copy(), "val": adata_mock.copy(), "test": adata_mock.copy()}
mock_cfg = {"phase9_normalization": {"target_sum": 10000, "log_transform": True}, 
            "paths": {"_plots": "output/"}} # Dummy path for the plot saver

try:
    # 1. Run Normalization
    mock_splits = p9.run_phase9(mock_splits, mock_cfg, logger)
    
    # 2. Extract Cache
    mock_cache = p9.extract_p9_data(mock_splits)
    
    # 3. Print Diagnostics
    p9.print_normalization_diagnostics(mock_cache, mock_cfg)
    
    # 4. Render Plot
    fig, path = splt.plot_phase9_normalization(mock_cache, mock_cfg)
    plt.show()
    
    print("\n✅ PHASE 9 CODE VERIFIED. It is safe to run on the main dataset.")

except Exception as e:
    print(f"\n❌ ERROR CAUGHT BEFORE MAIN RUN: {e}")